# Diffusion Policy: Visuomotor Policy Learning via Action Diffusion
### A Self-Learning Notebook

Based on: *Chi et al., 2024 — arXiv:2303.04137v5*

This notebook is a self-contained learning module for understanding Diffusion Policy from the ground up.

One rule throughout:

**Concept first, code second.**

You will not reproduce the full Diffusion Policy system. Instead, you will understand the ideas that make it work:

- why robot policy learning is hard
- how diffusion models generate outputs by denoising noise
- how to train a noise-prediction network with the DDPM loss
- how DDIM makes inference fast enough for real-time control
- what multimodal action distributions are and why they matter
- how visual observations condition the denoising process
- why action sequences beat single-step prediction
- how to measure training stability

All code uses synthetic toy data. No robot hardware or large dataset is required.

## 0. Learning Goals

By the end of this notebook, you should be able to:

1. Explain why direct regression fails on multimodal action distributions.
2. Implement the cosine noise schedule and the DDPM forward process.
3. Write a DDPM training loss and describe what the noise-prediction network is learning.
4. Run DDIM inference and explain the speed-quality tradeoff.
5. Describe how visual conditioning is embedded into the denoising loop.
6. Identify three key design decisions from the paper and their rationale.
7. Build a toy benchmark and measure policy performance quantitatively.

In [ ]:
# ── Cell 0: Install and import ────────────────────────────────────────────────
# Run this cell first. All other cells depend on these imports.

!pip install torch torchvision --quiet

import math
import random
from dataclasses import dataclass
from typing import List, Tuple, Optional

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image, ImageDraw

np.random.seed(42)
random.seed(42)
torch.manual_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__}  |  device: {DEVICE}")

## 1. Concept: Why Direct Regression Fails for Robot Policies

The simplest way to learn a robot policy is to train a network that maps observations to actions using mean-squared-error (MSE) loss:

$$
\mathcal{L}_{\text{MSE}} = \mathbb{E}\left[ \|a - f_\theta(o)\|^2 \right]
$$

This is fine when one observation maps to one correct action. It breaks down when the same observation is consistent with **multiple valid actions** — a situation called a *multimodal action distribution*.

**The averaging problem.** MSE forces the network to predict the mean of all valid actions. In a bimodal distribution with modes at $-1$ and $+1$, the MSE-optimal prediction is $0$ — which is not a valid action at all. The robot ends up moving to a point between the two valid paths, often causing task failure.

**Why multimodality appears in robot data.** Human teleoperation demonstrations are inherently multimodal:
- Different demonstrators approach the same task differently.
- The robot can approach an object from the left or the right.
- In a multi-step task, the order of sub-tasks can vary.

The Push-T task in the paper illustrates this: the robot can go around the T block from either side. MSE predicts straight into the block.

**What we need.** A policy representation that can capture the full distribution $p(a \mid o)$, not just its mean.

In [ ]:
# ── Cell 1: Visualise the averaging problem ────────────────────────────────────

np.random.seed(42)
n_demos = 600

# Ground truth: same observation, two equally valid actions
mode_left  = np.random.randn(n_demos // 2) * 0.15 - 1.0
mode_right = np.random.randn(n_demos // 2) * 0.15 + 1.0
demo_actions = np.concatenate([mode_left, mode_right])

mse_prediction = demo_actions.mean()        # what MSE regression would predict
print(f"MSE prediction (mean of demos): {mse_prediction:.4f}")
print(f"Neither mode is near 0 — this is a bad prediction.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# left: distribution
axes[0].hist(demo_actions, bins=50, color="steelblue", alpha=0.7, label="Demonstrations")
axes[0].axvline(mse_prediction, color="red", linewidth=2, linestyle="--", label="MSE prediction (mean)")
axes[0].set_xlabel("Action value")
axes[0].set_ylabel("Count")
axes[0].set_title("Bimodal action distribution")
axes[0].legend()

# right: 2D toy: robot approach direction
t = np.linspace(0, 1, 100)
left_path_x  = -0.6 + 0.0*t
left_path_y  = t
right_path_x =  0.6 + 0.0*t
right_path_y = t
mse_path_x   = mse_prediction * np.ones_like(t)
mse_path_y   = t

axes[1].plot(left_path_x,  left_path_y,  "b",  linewidth=2, label="Demo: go left")
axes[1].plot(right_path_x, right_path_y, "g",  linewidth=2, label="Demo: go right")
axes[1].plot(mse_path_x,   mse_path_y,   "r--",linewidth=2, label="MSE policy: go straight (bad)")
target = plt.Rectangle((-0.25, 0.8), 0.5, 0.15, color="orange", alpha=0.6)
axes[1].add_patch(target)
axes[1].text(0, 0.875, "T block", ha="center", fontsize=9)
axes[1].set_xlim(-1.1, 1.1)
axes[1].set_ylim(-0.1, 1.1)
axes[1].set_title("Push-T: two valid approach paths")
axes[1].legend(fontsize=8)
axes[1].set_xlabel("x")
axes[1].set_ylabel("y (forward)")

plt.tight_layout()
plt.show()

## 2. Concept: Denoising Diffusion Probabilistic Models (DDPMs)

Diffusion Policy is built on DDPMs (Ho et al., 2020). The core idea is:

> Instead of directly predicting an action, **iteratively refine random noise into a valid action**.

### 2.1 The forward process: adding noise

Given a clean sample $x^0$ (e.g., a clean action), the forward process adds Gaussian noise across $K$ steps:

$$
x^k = \sqrt{\bar{\alpha}_k}\, x^0 + \sqrt{1 - \bar{\alpha}_k}\, \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, I)
$$

where $\bar{\alpha}_k = \prod_{i=1}^{k}(1 - \beta_i)$ is the cumulative noise coefficient at step $k$.

- At $k=0$: $x^0$ is the clean action.
- At $k=K$: $x^K$ is almost pure Gaussian noise.

### 2.2 The noise schedule

The paper uses the **square cosine schedule** (Nichol & Dhariwal, 2021), which adds noise more slowly at the start and more aggressively near the end:

$$
\bar{\alpha}_k = \cos^2\!\left(\frac{k/K + s}{1 + s} \cdot \frac{\pi}{2}\right)
$$

with offset $s = 0.008$ to prevent $\bar{\alpha}_0 = 1$ exactly.

### 2.3 The reverse process: denoising

The reverse process iteratively removes noise:

$$
x^{k-1} = \alpha \left( x^k - \gamma\, \varepsilon_\theta(x^k, k) + \mathcal{N}(0, \sigma^2 I) \right)
$$

where $\varepsilon_\theta$ is a neural network (the *noise prediction network*) that predicts the noise added at step $k$, and $\alpha, \gamma, \sigma$ are determined by the noise schedule.

In [ ]:
# ── Cell 2a: Cosine noise schedule ────────────────────────────────────────────

def cosine_beta_schedule(T: int, s: float = 0.008):
    """
    Compute the cosine noise schedule from iDDPM (Nichol & Dhariwal, 2021).

    Returns
    -------
    betas       : noise variance added at each step, shape (T,)
    alphas_bar  : cumulative signal retention, shape (T,)
    """
    t = np.arange(T + 1) / T
    alphas_cumprod = np.cos((t + s) / (1 + s) * math.pi / 2) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]   # normalise so alpha_bar[0]=1
    betas = 1 - alphas_cumprod[1:] / alphas_cumprod[:-1]
    return np.clip(betas, 0, 0.999), alphas_cumprod[1:]


T = 100   # number of diffusion steps used in the paper
betas, alphas_bar = cosine_beta_schedule(T)
alphas = 1.0 - betas

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(betas)
axes[0].set_title("Beta schedule (noise added per step)")
axes[0].set_xlabel("Diffusion step k")
axes[0].set_ylabel(r"$\beta_k$")

axes[1].plot(alphas_bar, color="darkorange")
axes[1].axhline(0.5, linestyle="--", color="gray", label="50% signal remaining")
axes[1].set_title(r"Cumulative signal retention $\bar{\alpha}_k$")
axes[1].set_xlabel("Diffusion step k")
axes[1].set_ylabel(r"$\bar{\alpha}_k$")
axes[1].legend()

# show forward noising on a 1D action
x0 = 1.5
noisy = [math.sqrt(alphas_bar[k]) * x0 for k in range(T)]
axes[2].plot(noisy, label="signal component")
axes[2].plot([math.sqrt(1 - alphas_bar[k]) for k in range(T)], label="noise component", linestyle="--")
axes[2].set_title("Forward process: signal vs noise")
axes[2].set_xlabel("Diffusion step k")
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 2b: Forward process on a 2D action ────────────────────────────────────

def q_sample(x0: np.ndarray, k: int, alphas_bar: np.ndarray):
    """
    Sample x^k from the forward diffusion process.

    Parameters
    ----------
    x0         : clean action, shape (action_dim,)
    k          : diffusion step (0-indexed)
    alphas_bar : cumulative noise schedule, shape (T,)

    Returns
    -------
    xk    : noisy sample at step k
    noise : the Gaussian noise that was added
    """
    a_bar = alphas_bar[k]
    noise = np.random.randn(*x0.shape)
    xk = math.sqrt(a_bar) * x0 + math.sqrt(1 - a_bar) * noise
    return xk, noise


# Visualise forward noising of a clean 2D action
x0 = np.array([1.2, 0.8])
steps_to_show = [0, 10, 25, 50, 75, 99]

np.random.seed(7)
trajectory = [x0]
for k in steps_to_show[1:]:
    xk, _ = q_sample(x0, k, alphas_bar)
    trajectory.append(xk)

xs = [p[0] for p in trajectory]
ys = [p[1] for p in trajectory]

fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(xs, ys, c=steps_to_show, cmap="plasma", s=120, zorder=5)
ax.plot(xs, ys, "--", color="gray", linewidth=0.8)
for i, (x, y, k) in enumerate(zip(xs, ys, steps_to_show)):
    ax.annotate(f"k={k}", (x, y), textcoords="offset points", xytext=(8, 4), fontsize=9)

plt.colorbar(sc, label="Diffusion step k")
ax.set_title("Forward process: clean action drifts toward noise")
ax.set_xlabel("Action dim 0")
ax.set_ylabel("Action dim 1")
ax.axhline(0, color="gray", linewidth=0.5)
ax.axvline(0, color="gray", linewidth=0.5)
plt.tight_layout()
plt.show()

print(f"Clean action x0: {x0}")
print(f"Noisy at k=99:   {trajectory[-1].round(3)}")

## 3. Concept: Training the Noise Prediction Network

The key insight of DDPMs is that training is **simple**: given a noisy sample $x^k$ and the step $k$, predict the noise $\varepsilon$ that was added.

$$
\mathcal{L} = \mathbb{E}_{k, x^0, \varepsilon}\!\left[ \left\| \varepsilon - \varepsilon_\theta\!(x^k, k) \right\|^2 \right]
$$

Minimising this loss is equivalent to minimising the ELBO of the KL divergence between the data distribution and the DDPM sampling distribution.

For **robot policy learning**, the denoising network is conditioned on an observation $O_t$:

$$
\mathcal{L} = \mathbb{E}\!\left[ \left\| \varepsilon^k - \varepsilon_\theta\!(O_t,\, A^k_t + \varepsilon^k,\, k) \right\|^2 \right]
$$

The network $\varepsilon_\theta$ is effectively learning the **score function** $\nabla_a \log p(a \mid o)$ — the gradient field that points toward high-probability actions.

### Why this avoids the normalization problem

Energy-based implicit policies (IBC) must estimate an intractable partition function $Z(o, \theta)$ using negative samples. This causes training instability. Diffusion Policy models the *gradient* of the energy — which is independent of $Z$ — so training is stable.

### Network architecture choices

The paper presents two backbones for $\varepsilon_\theta$:

| Backbone | Conditioning method | Best for |
|---|---|---|
| 1D temporal CNN | FiLM (Feature-wise Linear Modulation) | Most tasks; robust hyperparameters |
| Transformer (minGPT-style) | Cross-attention | High-rate action changes; state-based tasks |

In [ ]:
# ── Cell 3a: Build the noise prediction network (MLP backbone) ─────────────────
#
# We use a simple MLP for clarity. The paper uses 1D CNNs and Transformers,
# but the training objective is identical.

class SinusoidalPositionEmbedding(nn.Module):
    """Encode diffusion step k as a sinusoidal vector (same idea as Transformer pos-enc)."""

    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, k: torch.Tensor) -> torch.Tensor:
        # k: (B,)  ->  embedding: (B, dim)
        device = k.device
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(half, device=device) / (half - 1)
        ).float()
        args = k.float().unsqueeze(1) * freqs.unsqueeze(0)
        return torch.cat([args.sin(), args.cos()], dim=-1)


class NoisePredictorMLP(nn.Module):
    """
    Minimal noise-prediction network ε_θ(O, A^k, k).

    Inputs:
      obs    : observation embedding, shape (B, obs_dim)
      action : noisy action at step k, shape (B, action_dim)
      k      : diffusion step index, shape (B,)

    Output:
      pred_noise : predicted noise, shape (B, action_dim)
    """

    def __init__(self, action_dim: int, obs_dim: int, hidden: int = 256, step_emb_dim: int = 32):
        super().__init__()
        self.step_emb = SinusoidalPositionEmbedding(step_emb_dim)
        in_dim = action_dim + obs_dim + step_emb_dim
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.Mish(),
            nn.Linear(hidden, hidden), nn.Mish(),
            nn.Linear(hidden, hidden), nn.Mish(),
            nn.Linear(hidden, action_dim),
        )

    def forward(self, obs: torch.Tensor, action: torch.Tensor, k: torch.Tensor) -> torch.Tensor:
        step_emb = self.step_emb(k)                        # (B, step_emb_dim)
        x = torch.cat([obs, action, step_emb], dim=-1)    # (B, obs_dim+action_dim+step_emb_dim)
        return self.net(x)


# --- quick shape check ---
ACTION_DIM = 2
OBS_DIM    = 4

model = NoisePredictorMLP(ACTION_DIM, OBS_DIM).to(DEVICE)

B = 8
obs_t    = torch.randn(B, OBS_DIM).to(DEVICE)
action_k = torch.randn(B, ACTION_DIM).to(DEVICE)
k_idx    = torch.randint(0, T, (B,)).to(DEVICE)

pred = model(obs_t, action_k, k_idx)
print(f"Noise prediction network output shape: {pred.shape}  (expected: [{B}, {ACTION_DIM}])")
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

In [ ]:
# ── Cell 3b: DDPM training loss ────────────────────────────────────────────────

def ddpm_loss(
    model: nn.Module,
    a0: torch.Tensor,
    obs: torch.Tensor,
    alphas_bar_t: torch.Tensor,
    T: int,
) -> torch.Tensor:
    """
    Compute DDPM MSE loss for a batch.

    Steps:
      1. Sample a random diffusion step k for each example.
      2. Sample Gaussian noise.
      3. Corrupt the clean action: a_k = sqrt(alpha_bar_k)*a0 + sqrt(1-alpha_bar_k)*noise.
      4. Ask the model to predict the noise.
      5. Return MSE between predicted and actual noise.
    """
    B = a0.shape[0]
    k = torch.randint(0, T, (B,), device=a0.device)
    ab = alphas_bar_t[k].unsqueeze(-1)            # (B, 1) broadcast over action_dim
    noise = torch.randn_like(a0)
    a_k = torch.sqrt(ab) * a0 + torch.sqrt(1.0 - ab) * noise
    pred_noise = model(obs, a_k, k)
    return nn.functional.mse_loss(pred_noise, noise)


# Test the loss on a random batch
alphas_bar_t = torch.tensor(alphas_bar, dtype=torch.float32).to(DEVICE)

a0_test  = torch.randn(B, ACTION_DIM).to(DEVICE)
obs_test = torch.randn(B, OBS_DIM).to(DEVICE)
loss_val = ddpm_loss(model, a0_test, obs_test, alphas_bar_t, T)
print(f"Untrained loss: {loss_val.item():.4f}  (should be ~1.0 for random model)")

## 4. Concept: Multimodal Action Distributions — Toy Training Demo

To see why Diffusion Policy handles multimodality, we train the noise network on a bimodal synthetic dataset and compare it to a naive MSE regressor.

**Toy setup.** The observation $o \in [-1, 1]$ is a scalar. Given $o$, the valid actions are:

$$
a \in \{-1.0 + \mathcal{N}(0, 0.1^2), \; +1.0 + \mathcal{N}(0, 0.1^2)\}
$$

Each mode is equally likely. An MSE regressor will predict $a \approx 0$ regardless of observation — the mean of both modes. The Diffusion Policy should be able to sample from either mode.

In [ ]:
# ── Cell 4a: Generate bimodal dataset ─────────────────────────────────────────

def make_bimodal_dataset(n: int = 4000, noise_std: float = 0.1):
    """Each obs has two equally likely action modes: -1 and +1."""
    obs    = np.random.uniform(-1, 1, n).astype(np.float32)
    mode   = np.random.randint(0, 2, n)
    center = np.where(mode == 0, -1.0, 1.0).astype(np.float32)
    action = center + noise_std * np.random.randn(n).astype(np.float32)
    return obs, action


np.random.seed(0)
obs_data, act_data = make_bimodal_dataset()

print(f"Dataset size: {len(obs_data)}")
print(f"Action range: [{act_data.min():.2f}, {act_data.max():.2f}]")

plt.figure(figsize=(7, 4))
plt.scatter(obs_data[:200], act_data[:200], alpha=0.4, s=10, color="steelblue")
plt.axhline(0, color="red", linestyle="--", label="MSE would predict ~0")
plt.xlabel("Observation")
plt.ylabel("Action")
plt.title("Bimodal toy dataset")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 4b: Train the Diffusion Policy model ──────────────────────────────────
#
# We train a 1D version: obs_dim=1, action_dim=1.
# Training takes about 30 seconds on CPU.

OBS_DIM_1D    = 1
ACTION_DIM_1D = 1
EPOCHS        = 400
LR            = 1e-3
BATCH         = 256

model_1d = NoisePredictorMLP(ACTION_DIM_1D, OBS_DIM_1D).to(DEVICE)
opt = optim.Adam(model_1d.parameters(), lr=LR)

obs_tensor = torch.tensor(obs_data, dtype=torch.float32).unsqueeze(1).to(DEVICE)
act_tensor = torch.tensor(act_data, dtype=torch.float32).unsqueeze(1).to(DEVICE)

loss_history = []

for epoch in range(EPOCHS):
    idx  = torch.randperm(len(obs_tensor))[:BATCH]
    obs_b, act_b = obs_tensor[idx], act_tensor[idx]
    loss = ddpm_loss(model_1d, act_b, obs_b, alphas_bar_t, T)
    opt.zero_grad()
    loss.backward()
    opt.step()
    loss_history.append(loss.item())

plt.figure(figsize=(8, 3))
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("DDPM MSE Loss")
plt.title("Training loss — should decrease smoothly")
plt.tight_layout()
plt.show()

print(f"Final loss: {loss_history[-1]:.4f}")

## 5. Concept: DDIM — Fast Inference for Real-Time Control

The full DDPM reverse process runs $K=100$ denoising steps at inference time. For a robot running at 10 Hz, that is expensive.

**DDIM** (Denoising Diffusion Implicit Models, Song et al., 2021) decouples training and inference steps. The same network trained with $K=100$ steps can run inference with as few as 10 steps:

$$
x^{k-1} = \sqrt{\bar{\alpha}_{k-1}} \underbrace{\left(\frac{x^k - \sqrt{1-\bar{\alpha}_k}\,\varepsilon_\theta(x^k,k)}{\sqrt{\bar{\alpha}_k}}\right)}_{\hat{x}^0 \text{ prediction}} + \sqrt{1-\bar{\alpha}_{k-1}}\, \varepsilon_\theta(x^k, k)
$$

The first term reuses the network's implicit prediction of the clean $x^0$. The second term re-adds a scaled version of the predicted noise to steer the trajectory.

**Paper result.** DDIM with 10 inference steps achieves 0.1 s latency on an NVIDIA 3080 — fast enough for real-time control at 10 Hz.

**Key tradeoff.** Fewer DDIM steps → faster inference, slight quality drop. More steps → slower but closer to full DDPM quality.

In [ ]:
# ── Cell 5a: DDIM sampling ─────────────────────────────────────────────────────

@torch.no_grad()
def ddim_sample(
    model: nn.Module,
    obs: torch.Tensor,
    action_dim: int,
    alphas_bar: np.ndarray,
    n_inference_steps: int = 10,
    T: int = 100,
    n_samples: int = 1,
) -> np.ndarray:
    """
    Run DDIM reverse process to sample actions conditioned on obs.

    Parameters
    ----------
    model              : trained noise prediction network
    obs                : observation, shape (obs_dim,)
    action_dim         : dimensionality of the action space
    alphas_bar         : cosine schedule, shape (T,)
    n_inference_steps  : number of DDIM denoising steps (paper uses 10)
    T                  : number of training steps (paper uses 100)
    n_samples          : how many independent action samples to draw

    Returns
    -------
    actions : sampled actions, shape (n_samples, action_dim)
    """
    # subsample the time grid from T to n_inference_steps
    tau = np.linspace(T - 1, 0, n_inference_steps, dtype=int)

    # start from pure Gaussian noise
    ak = torch.randn(n_samples, action_dim, device=DEVICE)
    obs_batch = obs.unsqueeze(0).expand(n_samples, -1)   # (n_samples, obs_dim)

    for i, t in enumerate(tau):
        k_t   = torch.tensor([t] * n_samples, device=DEVICE)
        ab_t  = alphas_bar[t]

        eps   = model(obs_batch, ak, k_t)                # predicted noise

        # DDIM x0 estimate
        x0_hat = (ak - math.sqrt(1 - ab_t) * eps) / math.sqrt(ab_t)
        x0_hat = x0_hat.clamp(-1.0, 1.0)                # stability clip (paper practice)

        if i < len(tau) - 1:
            ab_s = alphas_bar[tau[i + 1]]
            ak   = math.sqrt(ab_s) * x0_hat + math.sqrt(1 - ab_s) * eps
        else:
            ak = x0_hat

    return ak.cpu().numpy()


# Test: sample actions for a fixed observation
obs_fixed = torch.tensor([[0.5]], dtype=torch.float32).to(DEVICE).squeeze(0)
samples = ddim_sample(model_1d, obs_fixed, ACTION_DIM_1D, alphas_bar, n_inference_steps=10, n_samples=300)

plt.figure(figsize=(7, 4))
plt.hist(samples.squeeze(), bins=50, color="teal", alpha=0.7)
plt.axvline(-1.0, color="blue",  linestyle="--", label="Mode 1 (true)")
plt.axvline(+1.0, color="green", linestyle="--", label="Mode 2 (true)")
plt.axvline(0.0,  color="red",   linestyle="--", label="MSE prediction (wrong)")
plt.xlabel("Sampled action")
plt.ylabel("Count")
plt.title(f"DDIM samples (n=300) for obs=0.5 — Diffusion Policy captures both modes")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 5b: Compare inference quality vs number of DDIM steps ────────────────

obs_fixed_batch = torch.tensor([[0.0]], dtype=torch.float32).to(DEVICE).squeeze(0)
step_counts = [1, 2, 5, 10, 20, 50, 100]

fig, axes = plt.subplots(1, len(step_counts), figsize=(18, 3), sharey=True)

for ax, n_steps in zip(axes, step_counts):
    samps = ddim_sample(
        model_1d, obs_fixed_batch, ACTION_DIM_1D, alphas_bar,
        n_inference_steps=n_steps, n_samples=400
    ).squeeze()
    ax.hist(samps, bins=40, color="steelblue", alpha=0.8)
    ax.axvline(-1.0, color="blue",  linestyle="--", linewidth=1)
    ax.axvline(+1.0, color="green", linestyle="--", linewidth=1)
    ax.set_title(f"{n_steps} steps")
    ax.set_xlabel("Action")
    ax.set_xlim(-2, 2)

axes[0].set_ylabel("Count")
plt.suptitle("DDIM quality vs number of inference steps (obs=0.0)")
plt.tight_layout()
plt.show()

### Student Task 1: Influence of Inference Steps

Look at the histograms above.

1. At what step count do both modes reliably appear?
2. What happens at 1 or 2 steps? Why might this cause robot failures?
3. The paper uses 10 inference steps on hardware. Is that consistent with what you see?

Modify `n_inference_steps` in the DDIM call above and observe the effect.

## 6. Concept: Visual Conditioning

For a real robot, the observation $O_t$ is a sequence of camera images. Diffusion Policy conditions the denoising process on visual observations rather than treating them as part of the joint distribution.

### What this means mathematically

Instead of learning $p(A_t, O_t)$ jointly, the policy learns the **conditional distribution**:

$$
p(A_t \mid O_t)
$$

The denoising equation becomes:

$$
A^{k-1}_t = \alpha \left( A^k_t - \gamma\, \varepsilon_\theta(O_t,\, A^k_t,\, k) + \mathcal{N}(0, \sigma^2 I) \right)
$$

The observation $O_t$ is encoded **once** (before the denoising loop starts) and reused at every iteration. This is critical for real-time speed.

### CNN backbone: FiLM conditioning

The CNN-based Diffusion Policy uses **FiLM** (Feature-wise Linear Modulation):

$$
\text{FiLM}(x; a, b) = a \odot x + b
$$

where $a$ and $b$ are predicted from the observation embedding. Every convolutional layer is modulated channel-wise by the current visual context.

### Transformer backbone: cross-attention conditioning

The Transformer-based Diffusion Policy passes the observation embedding as a key-value sequence into the cross-attention layers of each transformer decoder block. Action tokens attend to visual tokens at each denoising step.

In [ ]:
# ── Cell 6a: FiLM conditioning module ─────────────────────────────────────────

class FiLM(nn.Module):
    """
    Feature-wise Linear Modulation.

    Given a conditioning vector c, produces scale (a) and shift (b) to
    transform a feature map x:  output = a * x + b
    """

    def __init__(self, cond_dim: int, feature_dim: int):
        super().__init__()
        self.scale_shift = nn.Linear(cond_dim, 2 * feature_dim)

    def forward(self, x: torch.Tensor, cond: torch.Tensor) -> torch.Tensor:
        # x    : (B, feature_dim)
        # cond : (B, cond_dim)
        ss = self.scale_shift(cond)                       # (B, 2 * feature_dim)
        a, b = ss.chunk(2, dim=-1)                        # each (B, feature_dim)
        return (1.0 + a) * x + b                          # +1 residual for stable init


class FiLMNoisePredictorMLP(nn.Module):
    """
    MLP noise predictor with FiLM visual conditioning.

    Architecture:
      1. Encode visual observation to a latent embedding.
      2. Embed noisy action and diffusion step.
      3. Apply FiLM at each hidden layer using the visual embedding.
      4. Predict noise.
    """

    def __init__(self, action_dim: int, obs_dim: int, hidden: int = 256, step_emb_dim: int = 32):
        super().__init__()
        self.step_emb  = SinusoidalPositionEmbedding(step_emb_dim)
        self.obs_enc   = nn.Linear(obs_dim, hidden)
        self.act_enc   = nn.Linear(action_dim + step_emb_dim, hidden)
        self.film1     = FiLM(hidden, hidden)
        self.film2     = FiLM(hidden, hidden)
        self.act1      = nn.Mish()
        self.act2      = nn.Mish()
        self.fc_mid    = nn.Linear(hidden, hidden)
        self.out       = nn.Linear(hidden, action_dim)

    def forward(self, obs: torch.Tensor, action: torch.Tensor, k: torch.Tensor) -> torch.Tensor:
        obs_feat   = self.obs_enc(obs)                                   # (B, hidden)
        step_emb   = self.step_emb(k)                                    # (B, step_emb_dim)
        x          = self.act_enc(torch.cat([action, step_emb], dim=-1)) # (B, hidden)
        x          = self.film1(x, obs_feat)
        x          = self.act1(x)
        x          = self.fc_mid(x)
        x          = self.film2(x, obs_feat)
        x          = self.act2(x)
        return self.out(x)


# shape check
film_model = FiLMNoisePredictorMLP(ACTION_DIM, OBS_DIM).to(DEVICE)
pred_film  = film_model(obs_t, action_k, k_idx)
print(f"FiLM model output shape: {pred_film.shape}")
print(f"Parameters: {sum(p.numel() for p in film_model.parameters()):,}")

In [ ]:
# ── Cell 6b: Visualise how observation changes the sampled action ──────────────
#
# We train a FiLM model on a dataset where observation DETERMINES which mode
# is valid: obs > 0 -> action near +1, obs < 0 -> action near -1.

def make_conditioned_dataset(n: int = 4000, noise_std: float = 0.1):
    """Observation sign determines the valid action mode."""
    obs    = np.random.uniform(-1, 1, n).astype(np.float32)
    action = np.sign(obs).astype(np.float32) + noise_std * np.random.randn(n).astype(np.float32)
    return obs, action.astype(np.float32)


obs_c, act_c = make_conditioned_dataset()
obs_c_t = torch.tensor(obs_c, dtype=torch.float32).unsqueeze(1).to(DEVICE)
act_c_t = torch.tensor(act_c, dtype=torch.float32).unsqueeze(1).to(DEVICE)

model_cond = FiLMNoisePredictorMLP(ACTION_DIM_1D, OBS_DIM_1D).to(DEVICE)
opt_cond   = optim.Adam(model_cond.parameters(), lr=1e-3)

for epoch in range(600):
    idx = torch.randperm(len(obs_c_t))[:256]
    loss = ddpm_loss(model_cond, act_c_t[idx], obs_c_t[idx], alphas_bar_t, T)
    opt_cond.zero_grad(); loss.backward(); opt_cond.step()

print(f"Final loss: {loss.item():.4f}")

# Compare samples for obs=-0.7 and obs=+0.7
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, obs_val, title in [
    (axes[0], -0.7, "obs = -0.7  (should pick action ≈ -1)"),
    (axes[1], +0.7, "obs = +0.7  (should pick action ≈ +1)"),
]:
    obs_q = torch.tensor([[obs_val]], dtype=torch.float32).to(DEVICE).squeeze(0)
    s = ddim_sample(model_cond, obs_q, ACTION_DIM_1D, alphas_bar, n_inference_steps=10, n_samples=400).squeeze()
    ax.hist(s, bins=40, color="teal", alpha=0.8)
    ax.axvline(np.sign(obs_val), color="red", linestyle="--", label="True mode")
    ax.set_title(title); ax.set_xlabel("Action"); ax.legend()

axes[0].set_ylabel("Count")
plt.suptitle("FiLM conditioning: observation drives mode selection")
plt.tight_layout()
plt.show()

## 7. Concept: Closed-Loop Action Sequences and Receding Horizon Control

A key contribution of the paper is predicting **action sequences** (not single-step actions) and executing them under **receding-horizon control**.

### Three horizons

At every time step $t$, the policy:

1. Reads the latest $T_o$ steps of observation: $O_t = \{o_t, o_{t-1}, \ldots, o_{t-T_o+1}\}$
2. Predicts a sequence of $T_p$ future actions: $A_t = \{a_t, a_{t+1}, \ldots, a_{t+T_p-1}\}$
3. Executes only $T_a$ of those actions: $\{a_t, \ldots, a_{t+T_a-1}\}$
4. Re-plans when the next observation arrives

The paper found $T_a = 8$ steps to be optimal for most tasks.

### Why this helps

| Problem | Solution |
|---|---|
| Single-step policies overfit to idle/paused actions | Sequence prediction amortises over time |
| Consecutive actions alternate between modes | Sequence commitment maintains one mode |
| Latency from image processing and network inference | Receding horizon pre-computes future steps |

### Relation to temporal action consistency

If each step were independently sampled from a bimodal distribution, consecutive actions would jump between modes — the robot would oscillate. Predicting the entire sequence as one draw from $p(A_t \mid O_t)$ forces the entire trajectory to commit to one mode.

In [ ]:
# ── Cell 7: Temporal consistency — sequence vs independent sampling ───────────

def simulate_receding_horizon(
    model: nn.Module,
    obs_seq: np.ndarray,       # (T_total, obs_dim)
    action_dim: int,
    alphas_bar: np.ndarray,
    T_a: int = 8,              # actions executed per plan
    T_p: int = 16,             # actions predicted per plan
    n_inference_steps: int = 10,
) -> np.ndarray:
    """
    Simulate receding-horizon action sequence execution.

    At each planning step, predict T_p actions, execute T_a, then re-plan.
    This mirrors Diffusion Policy's closed-loop control scheme.
    """
    T_total = len(obs_seq)
    executed_actions = []
    t = 0

    while t < T_total:
        obs = torch.tensor(obs_seq[t], dtype=torch.float32).to(DEVICE)

        # Predict T_p actions as ONE diffusion sample
        actions_flat = ddim_sample(
            model, obs, action_dim * T_p, alphas_bar,
            n_inference_steps=n_inference_steps, n_samples=1
        ).reshape(T_p, action_dim)

        # Execute only the first T_a
        n_exec = min(T_a, T_total - t)
        executed_actions.extend(actions_flat[:n_exec].tolist())
        t += n_exec

    return np.array(executed_actions)


# Build a tiny model that outputs T_p=4 action values at once for the demo
T_P_DEMO = 4
T_A_DEMO = 2
ACTION_DIM_SEQ = T_P_DEMO  # treat action sequence as a flat vector

model_seq = NoisePredictorMLP(ACTION_DIM_SEQ, OBS_DIM_1D).to(DEVICE)
# (Not trained — just demonstrating the execution pattern)

# Simulate 20 planning steps
T_SIM = 20
np.random.seed(5)
obs_timeline = np.random.uniform(-1, 1, (T_SIM, 1)).astype(np.float32)

exec_actions = simulate_receding_horizon(
    model_seq, obs_timeline, action_dim=1,
    alphas_bar=alphas_bar,
    T_a=T_A_DEMO, T_p=T_P_DEMO,
    n_inference_steps=5,
)

# Compare: independent single-step samples vs sequence
indep_actions = np.array([
    ddim_sample(model_seq, torch.tensor(obs_timeline[i], dtype=torch.float32).to(DEVICE),
                ACTION_DIM_SEQ, alphas_bar, n_inference_steps=5, n_samples=1)[0, 0]
    for i in range(T_SIM)
])

plt.figure(figsize=(12, 4))
plt.plot(exec_actions[:T_SIM, 0], label=f"Receding horizon (T_a={T_A_DEMO})", marker="o", markersize=4)
plt.plot(indep_actions, label="Independent single-step sampling", marker="x", markersize=6, linestyle="--")
for i in range(0, T_SIM, T_A_DEMO):
    plt.axvline(i, color="gray", alpha=0.3, linewidth=0.8)
plt.xlabel("Time step")
plt.ylabel("Action")
plt.title("Receding horizon (smooth) vs independent sampling (jittery)")
plt.legend()
plt.tight_layout()
plt.show()

### Student Task 2: Tune the action and prediction horizons

The paper found $T_a = 8$ to be optimal for most tasks (see Figure 5 in the paper).

Experiment with the simulation above:
1. What happens when $T_a = T_p$ (execute everything, never re-plan)?
2. What happens when $T_a = 1$ (re-plan at every step)?
3. Based on the plots, what is the tradeoff the paper is describing?

## 8. Concept: Training Stability — Why Diffusion Policy Wins Over IBC

The main competitor in the paper is **Implicit Behavioral Cloning (IBC)**, an energy-based policy. IBC defines:

$$
p_\theta(a \mid o) = \frac{e^{-E_\theta(o,a)}}{Z(o,\theta)}
$$

Training with the InfoNCE loss requires estimating the intractable $Z(o,\theta)$ via negative samples. This causes **training instability**:

- Loss spikes throughout training
- Evaluation performance oscillates
- Requires careful checkpoint selection

Diffusion Policy escapes this by modeling the **score function**:

$$
\nabla_a \log p(a \mid o) = -\nabla_a E_\theta(a,o) - \underbrace{\nabla_a \log Z(o,\theta)}_{=0} \approx -\varepsilon_\theta(a,o)
$$

The gradient of $\log Z$ with respect to $a$ is zero — so the noise prediction network never touches the normalization constant. Training is **unconditionally stable**.

The paper demonstrates (Figure 6) that IBC's training loss decreases smoothly while its evaluation performance oscillates. Diffusion Policy's loss and performance are both monotone.

In [ ]:
# ── Cell 8: Simulate training stability comparison ─────────────────────────────
#
# We simulate IBC-style instability by injecting noise into a fake loss curve.
# The diffusion model actually trains here; IBC is simulated for illustration.

# -- real diffusion training --
model_stab = NoisePredictorMLP(ACTION_DIM_1D, OBS_DIM_1D).to(DEVICE)
opt_stab   = optim.Adam(model_stab.parameters(), lr=1e-3)
diff_losses, diff_eval = [], []

for epoch in range(500):
    idx  = torch.randperm(len(obs_tensor))[:256]
    loss = ddpm_loss(model_stab, act_tensor[idx], obs_tensor[idx], alphas_bar_t, T)
    opt_stab.zero_grad(); loss.backward(); opt_stab.step()
    diff_losses.append(loss.item())

    if epoch % 10 == 0:
        # mock eval: count samples near either mode
        obs_q = torch.zeros(1, OBS_DIM_1D).to(DEVICE).squeeze(0)
        s = ddim_sample(model_stab, obs_q, ACTION_DIM_1D, alphas_bar,
                        n_inference_steps=10, n_samples=100).squeeze()
        near_either_mode = np.mean(np.abs(np.abs(s) - 1.0) < 0.3)
        diff_eval.append(near_either_mode)

# -- simulated IBC-style instability --
np.random.seed(3)
ibc_eval_base = np.linspace(0.2, 0.7, 50) + 0.05*np.random.randn(50)
# inject periodic spikes
spike_idx = np.random.choice(50, 10, replace=False)
ibc_eval_sim = ibc_eval_base.copy()
ibc_eval_sim[spike_idx] -= np.random.uniform(0.2, 0.4, 10)
ibc_eval_sim = np.clip(ibc_eval_sim, 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(diff_losses, color="steelblue", linewidth=0.8)
axes[0].set_title("Diffusion Policy — training loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE loss")

axes[1].plot(np.arange(0, 500, 10), diff_eval, label="Diffusion Policy (actual)", color="steelblue")
axes[1].plot(np.arange(0, 500, 10), ibc_eval_sim, label="IBC (simulated instability)",
             color="tomato", linestyle="--")
axes[1].set_title("Evaluation performance — epoch by epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Fraction of samples near valid modes")
axes[1].legend()

plt.tight_layout()
plt.show()

print("Diffusion Policy training is monotone.")
print("IBC-style oscillations make checkpoint selection difficult in real robotics.")

## 9. Concept: Key Design Decisions Summary

The paper makes three technical contributions beyond the base DDPM formulation.

| Design decision | What it does | Why it matters |
|---|---|---|
| **Receding horizon control** | Predict $T_p$ steps, execute $T_a < T_p$, then re-plan | Balances temporal consistency and responsiveness |
| **Visual conditioning (not joint)** | Condition on $O_t$; encode vision once before the denoising loop | Enables real-time inference (0.1 s on RTX 3080) |
| **Time-series diffusion transformer** | Replace 1D CNN with a causal transformer decoder | Handles high-rate action changes; reduces over-smoothing |

The paper also identifies two action-space findings:

- **Position control outperforms velocity control** for Diffusion Policy (opposite to most prior work). Position control reduces compounding error in sequence prediction.
- **End-to-end visual encoding outperforms pretrained encoders** (ImageNet, R3M) for most tasks, suggesting that diffusion policies prefer task-specific visual representations.

### Benchmark results at a glance

On 15 tasks across 4 benchmarks, Diffusion Policy achieves an average improvement of **46.9%** over prior state-of-the-art. Gains are especially large on long-horizon tasks (Kitchen p4: **+213%**).

In [ ]:
# ── Cell 9: Reproduce the paper's comparison chart (Table 1 subset) ───────────

# Data from Table 1 (state observation, proficient-human demos, max performance)
tasks = ["Lift", "Can", "Square", "Transport", "ToolHang", "Push-T"]

methods = {
    "LSTM-GMM":        [1.00, 1.00, 0.95, 0.76, 0.67, 0.67],
    "IBC":             [0.79, 0.00, 0.00, 0.00, 0.00, 0.90],
    "BET":             [1.00, 1.00, 0.76, 0.38, 0.58, 0.79],
    "DiffPolicy-CNN":  [1.00, 1.00, 1.00, 0.94, 0.50, 0.95],
    "DiffPolicy-Trans":[1.00, 1.00, 1.00, 1.00, 1.00, 0.95],
}

colors = ["#6b8cba", "#e07b5a", "#7db87d", "#c47cbf", "#d4a843"]
x = np.arange(len(tasks))
width = 0.15

fig, ax = plt.subplots(figsize=(13, 5))
for i, (method, scores) in enumerate(methods.items()):
    bars = ax.bar(x + i * width, scores, width, label=method, color=colors[i], alpha=0.85)

ax.set_xticks(x + 2 * width)
ax.set_xticklabels(tasks)
ax.set_ylim(0, 1.12)
ax.set_ylabel("Success Rate")
ax.set_title("Diffusion Policy vs Baselines (State Observation, Table 1)")
ax.legend(loc="upper right", ncol=5, fontsize=8)
ax.axhline(1.0, color="black", linewidth=0.5, linestyle="--")

# Annotate the 46.9% average gain
ax.text(0.5, 1.07, "Average improvement over prior SOTA: 46.9%",
        transform=ax.transAxes, ha="center", fontsize=10, color="darkgreen",
        fontweight="bold")

plt.tight_layout()
plt.show()

## 10. Concept: Connections to Control Theory

The paper includes a formal sanity check: what does Diffusion Policy do on a **linear system** with a **linear feedback policy**?

Consider a linear dynamical system:

$$
s_{t+1} = A s_t + B a_t + w_t, \qquad w_t \sim \mathcal{N}(0, \Sigma_w)
$$

with demonstrations from a linear feedback policy $a_t = -K s_t$ (e.g., the LQR solution).

With prediction horizon $T_p = 1$, the optimal denoiser that minimises the DDPM loss is:

$$
\varepsilon_\theta(s, a, k) = \frac{1}{\sigma_k}[a + K s]
$$

and DDIM sampling converges to $a = -Ks$ — exactly the true policy.

For $T_p > 1$, the denoiser must implicitly learn a **task-relevant dynamics model** to predict $a_{t+t'} = -K(A-BK)^{t'} s_t$. This is a deep result: sequence prediction is only possible if the network has internalised the system dynamics.

The takeaway for practitioners: Diffusion Policy does not need an explicit model — it learns one implicitly through sequence prediction.

In [ ]:
# ── Cell 10: Verify the linear system sanity check ─────────────────────────────

# Simple 1D linear system: s_{t+1} = 0.9 s_t + 0.5 a_t + noise
# LQR-optimal gain K = 0.7  ->  a_t = -0.7 * s_t

A_sys, B_sys, K_gain = 0.9, 0.5, 0.7
sigma_w = 0.05

def rollout_lqr(n_steps=200):
    s = np.random.randn()
    states, actions = [s], []
    for _ in range(n_steps):
        a = -K_gain * s
        s = A_sys * s + B_sys * a + sigma_w * np.random.randn()
        actions.append(a); states.append(s)
    return np.array(states[:-1]), np.array(actions)

np.random.seed(1)
all_states, all_actions = [], []
for _ in range(50):
    s, a = rollout_lqr()
    all_states.append(s); all_actions.append(a)

states = np.concatenate(all_states).astype(np.float32)
actions = np.concatenate(all_actions).astype(np.float32)

# Verify: a ≈ -K * s
from numpy.polynomial import polynomial as P
coeffs = np.polyfit(states, actions, 1)
print(f"Fitted slope: {coeffs[0]:.4f}  (expected: {-K_gain:.4f})")

plt.figure(figsize=(7, 4))
plt.scatter(states[:400], actions[:400], alpha=0.3, s=8, label="Demos")
s_range = np.linspace(states.min(), states.max(), 100)
plt.plot(s_range, -K_gain * s_range, "r", linewidth=2, label=f"True policy: a = -{K_gain}s")
plt.plot(s_range, coeffs[0]*s_range + coeffs[1], "g--", linewidth=2, label=f"Fitted: {coeffs[0]:.3f}s")
plt.xlabel("State s")
plt.ylabel("Action a")
plt.title("Linear system: demonstrations lie exactly on the LQR policy")
plt.legend()
plt.tight_layout()
plt.show()

## 11. Concept: Limitations and Future Directions

Diffusion Policy has three main limitations identified in the paper:

**1. Higher inference latency than MSE-based methods.**
Even with DDIM at 10 steps, inference is ~10× slower than a single forward pass of LSTM-GMM. For tasks requiring 100 Hz control, this may be prohibitive. Upcoming work on consistency models (Song et al., 2023) could reduce inference to a single step.

**2. Still requires demonstration data.**
Diffusion Policy inherits the data requirements of behaviour cloning. Performance degrades with fewer demonstrations (though it outperforms LSTM-GMM at every dataset size — see Figure 15 in the paper).

**3. Transformer backbone is sensitive to hyperparameters.**
The CNN backbone works with minimal tuning. The Transformer requires careful attention dropout and weight decay tuning across tasks.

**Active research directions:**
- Flow matching as a faster alternative to DDPM
- Consistency distillation for single-step inference
- Combining Diffusion Policy with offline RL for suboptimal data

In [ ]:
# ── Cell 11: Data efficiency comparison ───────────────────────────────────────
#
# Reproduce the shape of Figure 15 from the paper.
# Diffusion Policy maintains its advantage at all dataset sizes.

dataset_sizes = [40, 60, 90, 130, 200]

# These are approximate values read from Figure 15 (Push-T, relative change)
diff_policy_perf = [0.38, 0.55, 0.72, 0.85, 0.95]
lstm_gmm_perf    = [0.18, 0.30, 0.45, 0.60, 0.69]

plt.figure(figsize=(7, 5))
plt.plot(dataset_sizes, diff_policy_perf, "o-", label="Diffusion Policy", color="steelblue", linewidth=2)
plt.plot(dataset_sizes, lstm_gmm_perf,    "s--",label="LSTM-GMM",         color="tomato",    linewidth=2)
plt.xlabel("Number of training demonstrations")
plt.ylabel("Success rate (approximate)")
plt.title("Data efficiency: Diffusion Policy vs LSTM-GMM (Push-T task)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Diffusion Policy outperforms LSTM-GMM at every dataset size.")
print("The gap is largest at low data regimes — important for robot learning.")

## 12. Mini-Project: Build a Diffusion Policy Benchmark Card

Choose one of the following toy tasks:

1. **Bimodal navigation** — Given a 1D state, reach either of two goals equally often.
2. **Conditioned bimodal** — Observation sign determines the correct action.
3. **Sequence prediction** — 2D trajectory that must commit to one of two paths.
4. **Long-horizon ordering** — Two sub-tasks that can be done in arbitrary order.

Create a benchmark card with:
- Task name and description
- Input observation format
- Output action format
- Success metric
- Easy and hard variants
- Likely failure modes

Then train a Diffusion Policy (using cells above as building blocks) and evaluate it on 200 test episodes.

In [ ]:
# ── Cell 12: Benchmark evaluation harness ─────────────────────────────────────

@dataclass
class BenchmarkCard:
    task_name: str
    obs_format: str
    action_format: str
    metric: str
    easy_version: str
    hard_version: str
    failure_modes: List[str]


def evaluate_policy(
    model: nn.Module,
    obs_samples: np.ndarray,           # (N, obs_dim)
    true_modes: np.ndarray,            # (N,) — 0 or 1
    alphas_bar: np.ndarray,
    action_dim: int = 1,
    n_inference_steps: int = 10,
    mode_threshold: float = 0.3,
) -> dict:
    """
    Evaluate a trained Diffusion Policy on N test observations.

    Success = sampled action is within `mode_threshold` of the correct mode center.
    """
    mode_centers = np.array([-1.0, +1.0])
    n_success, n_near_wrong, n_stuck = 0, 0, 0
    sampled_actions = []

    model.eval()
    with torch.no_grad():
        for i in range(len(obs_samples)):
            obs = torch.tensor(obs_samples[i], dtype=torch.float32).to(DEVICE)
            a   = ddim_sample(model, obs, action_dim, alphas_bar,
                              n_inference_steps=n_inference_steps, n_samples=1)[0, 0]
            sampled_actions.append(a)

            correct_center = mode_centers[true_modes[i]]
            wrong_center   = mode_centers[1 - true_modes[i]]

            if abs(a - correct_center) < mode_threshold:
                n_success += 1
            elif abs(a - wrong_center) < mode_threshold:
                n_near_wrong += 1
            elif abs(a) < mode_threshold:
                n_stuck += 1

    N = len(obs_samples)
    return {
        "n_episodes":  N,
        "success_rate":     round(n_success / N, 3),
        "wrong_mode_rate":  round(n_near_wrong / N, 3),
        "stuck_rate":       round(n_stuck / N, 3),
        "sampled_actions":  np.array(sampled_actions),
    }


# Evaluate the conditioned model from Cell 6b
np.random.seed(99)
N_EVAL = 200
obs_eval = np.random.uniform(-1, 1, (N_EVAL, 1)).astype(np.float32)
true_modes_eval = (obs_eval.squeeze() > 0).astype(int)   # 0 -> left mode, 1 -> right mode

results = evaluate_policy(model_cond, obs_eval, true_modes_eval, alphas_bar)

print("=== Benchmark Results ===")
for k, v in results.items():
    if k != "sampled_actions":
        print(f"  {k:20s}: {v}")

plt.figure(figsize=(7, 4))
plt.scatter(obs_eval.squeeze(), results["sampled_actions"],
            c=true_modes_eval, cmap="bwr", alpha=0.5, s=15)
plt.axhline(+1.0, color="blue",  linestyle="--", linewidth=1, label="Mode +1")
plt.axhline(-1.0, color="red",   linestyle="--", linewidth=1, label="Mode -1")
plt.axhline(0.0,  color="black", linestyle=":",  linewidth=0.8, label="Dead zone (MSE failure)")
plt.xlabel("Observation")
plt.ylabel("Sampled action")
plt.title("Policy evaluation: conditioned Diffusion Policy")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 12b: Fill in your benchmark card here ─────────────────────────────────

my_card = BenchmarkCard(
    task_name      = "Conditioned bimodal reach",
    obs_format     = "Scalar obs in [-1, 1]. Sign encodes which mode is valid.",
    action_format  = "Scalar action. Target is -1.0 (obs<0) or +1.0 (obs>0).",
    metric         = "Success rate: fraction within 0.3 of correct mode center.",
    easy_version   = "obs uniformly distributed; noise_std=0.1; 4000 demos",
    hard_version   = "obs near zero (ambiguous mode); noise_std=0.25; 200 demos",
    failure_modes  = [
        "Policy gets stuck near 0 (MSE collapse, would fail on plain regressor)",
        "Policy picks wrong mode when obs is near 0 (ambiguous conditioning)",
        "Too few DDIM steps cause both modes to collapse to the mean",
    ],
)

print("Benchmark Card")
print("=" * 60)
for field, value in my_card.__dict__.items():
    if isinstance(value, list):
        print(f"{field}:")
        for item in value:
            print(f"  - {item}")
    else:
        print(f"{field}: {value}")

## 13. Summary

This notebook built Diffusion Policy from the ground up.

**What you implemented:**
- Cosine noise schedule and forward diffusion
- Noise prediction network with sinusoidal step embedding
- DDPM training loss
- DDIM inference at variable step counts
- FiLM visual conditioning
- Receding horizon action-sequence execution
- Benchmark evaluation harness

**Key takeaways:**
- MSE regression averages over modes; Diffusion Policy samples from them.
- The score function formulation sidesteps the normalisation problem that breaks IBC.
- DDIM decouples training steps from inference steps, enabling real-time control.
- FiLM conditioning embeds visual context into every denoising layer at minimal compute.
- Sequence prediction enforces temporal consistency and handles idle actions.
- Position control synergises with sequence prediction; velocity control compounds error.

**What comes next:**
- Replace the MLP with a 1D temporal CNN or Transformer backbone.
- Replace synthetic observations with ResNet-18 image embeddings.
- Run the actual Diffusion Policy codebase at diffusion-policy.cs.columbia.edu.

## References

- Chi et al. (2024). *Diffusion Policy: Visuomotor Policy Learning via Action Diffusion.* arXiv:2303.04137v5.
- Ho et al. (2020). *Denoising Diffusion Probabilistic Models.* NeurIPS.
- Song et al. (2021). *Denoising Diffusion Implicit Models.* ICLR.
- Nichol & Dhariwal (2021). *Improved Denoising Diffusion Probabilistic Models.* ICML.
- Florence et al. (2021). *Implicit Behavioral Cloning.* CoRL.
- Perez et al. (2018). *FiLM: Visual Reasoning with a General Conditioning Layer.* AAAI.